In [4]:
import sys
import os

# Set working directory to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from src.data.spark_pipeline import get_spark_session
from src.data.databricks_client import DatabricksClient
from src.models.regression import StockPriceRegressor
from src.models.classification import StockClassifier
from src.models.explainability import ModelExplainer

print("Phase 3 imports successful!")

Phase 3 imports successful!


In [5]:
# 1. Initialize sessions & client
spark = get_spark_session()
db_client = DatabricksClient()

# 2. Read stored indicators and sentiment tables
indicators_df = db_client.read_dataset(spark, table_name="aapl_indicators").toPandas()
sentiment_df = db_client.read_dataset(spark, table_name="aapl_sentiment").toPandas()

# 3. Convert timestamps and align datasets on date
indicators_df["date"] = pd.to_datetime(indicators_df["date"])
sentiment_df["date"] = pd.to_datetime(sentiment_df["timestamp"]).dt.date
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

# Aggregate daily sentiment score averages
daily_sentiment = (
    sentiment_df.groupby(["date", "ticker"])[
        ["neg_score", "neu_score", "pos_score", "compound_score"]
    ]
    .mean()
    .reset_index()
)

# Merge Technical Features with Sentiment Features
df = pd.merge(indicators_df, daily_sentiment, on=["date", "ticker"], how="left").fillna(0)
df.head()

[2026-09-22 15:42:00] [INFO] [stonks_maker]: Reading dataset locally from data/processed/aapl_indicators
[2026-09-22 15:42:00] [INFO] [stonks_maker]: Reading dataset locally from data/processed/aapl_sentiment


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/notebooks/data/processed/aapl_sentiment.

In [ ]:
# --- 1. Train Target Price Regressor ---
regressor = StockPriceRegressor()
X_reg, y_reg = regressor.prepare_data(df, target_horizon=1)

# Split Train/Test (Time Series split)
split_idx = int(len(X_reg) * 0.8)
X_train_r, X_test_r = X_reg.iloc[:split_idx], X_reg.iloc[split_idx:]
y_train_r, y_test_r = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]

regressor.train(X_train_r, y_train_r)
reg_metrics = regressor.evaluate(X_test_r, y_test_r)
regressor.save_model("models/regressor.joblib")

# --- 2. Train Classifiers ---
classifier = StockClassifier()
prepared_clf_df = classifier.prepare_data(df)

feature_cols = regressor.feature_names
X_clf = prepared_clf_df[feature_cols]
y_dir = prepared_clf_df["target_direction"]
y_strat = prepared_clf_df["target_strategy"]

X_train_c, X_test_c = X_clf.iloc[:split_idx], X_clf.iloc[split_idx:]
classifier.train_direction(X_train_c, y_dir.iloc[:split_idx])
classifier.train_strategy(X_train_c, y_strat.iloc[:split_idx])

clf_metrics = classifier.evaluate(X_test_c, y_dir.iloc[split_idx:], y_strat.iloc[split_idx:])
classifier.save_models()

print("Model Training Complete!")

In [ ]:
# Compute SHAP explanation on the latest sample instance
sample_instance = X_test_r.tail(1)
explainer = ModelExplainer(regressor.model, regressor.feature_names)
explanation = explainer.explain_instance(sample_instance)

print("SHAP Feature Contributions for Latest Prediction:")
print(explanation)